# QuranMB.v2 ASR Evaluation — Multi-Model Benchmark (Colab)

Evaluates 7 ASR models on `IqraEval/QuranMB.v2` (test split) using phoneme-level PER and BLEU.


In [3]:
# ════════════════════════════════════════════════════════════════════
# CELL 1 — numpy pin
# ════════════════════════════════════════════════════════════════════
!pip install "numpy>=2.2,<2.3" "pyarrow>=20.0.0" --no-deps -q

# ════════════════════════════════════════════════════════════════════
# CELL 2 — system packages
# ════════════════════════════════════════════════════════════════════
!apt-get update -qq
!apt-get install -y -qq ffmpeg libsndfile1

# ════════════════════════════════════════════════════════════════════
# CELL 3 — ALL pip installs EXCEPT torch/torchvision/torchaudio.
# Every line uses --no-deps. Without it, a sub-dependency's loose torch
# pin silently upgrades torch and breaks the ABI chain (happened 4x).
# pyarrow is deliberately excluded from the batch below — Cell 1 already
# owns that pin; a second unpinned install here would silently clobber it.
# ════════════════════════════════════════════════════════════════════
!pip install fairseq2 \
  --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.8.0/cu128 \
  --trusted-host fair.pkg.atmeta.com \
  --no-deps -q

!pip install fairseq2n==0.8.1 --no-deps -q

!pip install omnilingual-asr qwen-asr --no-deps -q

!pip install \
  retrying s3fs psutil rich ruamel.yaml tiktoken torcheval wandb \
  clusterscope editdistance huggingface_hub importlib_metadata importlib_resources \
  mypy-extensions packaging safetensors sacrebleu tensorboard transformers \
  typing_extensions click click-option-group aiobotocore aiohttp fsspec \
  --no-deps -q

!pip install jiwer sacrebleu datasets pandas mishkal -q

# ════════════════════════════════════════════════════════════════════
# CELL 4 — torch/torchvision/torchaudio, together, LAST.
# No pip install runs after this cell for the rest of the session.
# ════════════════════════════════════════════════════════════════════
!pip install torch==2.8.0+cu128 torchvision==0.23.0+cu128 torchaudio==2.8.0+cu128 \
  --index-url https://download.pytorch.org/whl/cu128 \
  --no-deps --force-reinstall -q

!pip install \
  nvidia-cublas-cu12==12.8.4.1 \
  nvidia-cuda-cupti-cu12==12.8.90 \
  nvidia-cuda-nvrtc-cu12==12.8.93 \
  nvidia-cuda-runtime-cu12==12.8.90 \
  nvidia-cudnn-cu12==9.10.2.21 \
  nvidia-cufft-cu12==11.3.3.83 \
  nvidia-cufile-cu12==1.13.1.3 \
  nvidia-curand-cu12==10.3.9.90 \
  nvidia-cusolver-cu12==11.7.3.90 \
  nvidia-cusparse-cu12==12.5.8.93 \
  nvidia-cusparselt-cu12==0.7.1 \
  nvidia-nccl-cu12==2.27.3 \
  nvidia-nvjitlink-cu12==12.8.93 \
  nvidia-nvtx-cu12==12.8.90 \
  --no-deps -q

# ════════════════════════════════════════════════════════════════════
# CELL 5 — verification. If torch is not 2.8.0, STOP here.
# ════════════════════════════════════════════════════════════════════
import torch
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda)
assert torch.__version__.startswith("2.8.0"), \
    f"torch was silently upgraded to {torch.__version__} — DO NOT PROCEED"

import torchaudio
print("torchaudio:", torchaudio.__version__)

import fairseq2
import fairseq2n
from fairseq2.data import data_pipeline
print("fairseq2 + fairseq2n OK (C++ bindings loaded)")

import omnilingual_asr
print("omnilingual_asr:", omnilingual_asr.__version__)

print("\n✓ All ABI-sensitive imports verified clean.")

# ════════════════════════════════════════════════════════════════════
# CELL 6 — HuggingFace token
# ════════════════════════════════════════════════════════════════════
import os
from google.colab import userdata

COLAB_WORKING = "/content"

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
    print("HF_TOKEN loaded from Colab Secrets.")
except Exception:
    print("WARNING: HF_TOKEN not found — add it via the 🔑 Secrets panel (left sidebar). "
          "Gated Gemma 4 models will fail without it.")

# ════════════════════════════════════════════════════════════════════
# CELL 7 — phonemizer setup
# ════════════════════════════════════════════════════════════════════
import sys, re, gc, tempfile, warnings, subprocess

if not os.path.exists("/content/MSA_phonetiser"):
    subprocess.run(
        ["git", "clone", "https://github.com/Iqra-Eval/MSA_phonetiser.git", "/content/MSA_phonetiser"],
        check=True,
    )

os.chdir("/content/MSA_phonetiser")
sys.path.insert(0, os.getcwd())

from phonetiser.phonetise_Arabic import phonetise
from mishkal.tashkeel import TashkeelClass

tashkeel = TashkeelClass()

def arabic_to_phonemes(arabic_text: str) -> str:
    diacritised = tashkeel.tashkeel(arabic_text)
    result = phonetise(diacritised)
    if not result or not result[1]:
        return ""
    phonemes = result[1][0]
    phonemes = phonemes.replace(" sil", "").replace("sil", "").strip()
    phonemes = re.sub(r'(\w+)[01]', r'\1', phonemes)
    return phonemes

# ════════════════════════════════════════════════════════════════════
# CELL 8 — analysis imports
# ════════════════════════════════════════════════════════════════════
"""
ASR Evaluation on Quran-Ayah-Corpus (QuranMB.v2)
Metrics: PER (phoneme error rate) and BLEU
"""
import numpy as np
import soundfile as sf
import pandas as pd
from datasets import load_dataset, Audio
import jiwer
import sacrebleu
import unicodedata

warnings.filterwarnings("ignore")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
torch: 2.8.0+cu128 | CUDA: 12.8
torchaudio: 2.8.0+cu128
fairseq2 + fairseq2n OK (C++ bindings loaded)
omnilingual_asr: 0.1.0

✓ All ABI-sensitive imports verified clean.


In [1]:
!pip install "bitsandbytes>=0.46.1" --no-deps -q

import importlib
try:
    import bitsandbytes as bnb
    print("bitsandbytes version:", bnb.__version__)
    print("CUDA setup OK")
except Exception as e:
    import traceback
    traceback.print_exc()

bitsandbytes version: 0.49.2
CUDA setup OK


In [2]:
from transformers.utils import is_bitsandbytes_available
print(is_bitsandbytes_available())

True


In [15]:
print(arabic_to_phonemes("إِنَّ اللَّهَ بِكُلِّ شَيْءٍ عَلِيمٌ"))

phoetising utterance
 <iion~ Alla~ha bikul~ $ayo'K EaliymN
['<', 'i0', 'nn']
skipped
['<', 'i0', 'nn']
['l', 'l', 'aa', 'h', 'a']
['b', 'i0', 'k', 'u0', 'll']
skipped
['b', 'i0', 'k', 'u0', 'll']
['$', 'a', 'y', '<', 'i1', 'n']
['E', 'a', 'l', 'ii0', 'm', 'u1', 'n']
< i nn l l aa h a b i k u ll $ a y < i n E a l ii m u n


In [16]:
#CONFIG  (Colab: T4/A100 GPU)
DATASET_ID  = "IqraEval/QuranMB.v2"
REF_DATASET = "IqraEval/test_references"
SPLIT       = "test"
NUM_SAMPLES = 5          # set to 1640 for full test set
TARGET_SR   = 16_000
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cpu":
    print("WARNING: No GPU detected — set Runtime → Change runtime type → T4 GPU.")

# Each entry: HuggingFace model_id, backend family, optional extra kwargs
MODELS = [
    # {"model_id": "google/gemma-4-E4B-it", "family": "gemma"},
     {"model_id": "google/gemma-4-E2B-it", "family": "gemma"},
    # {"model_id": "Qwen/Qwen3-ASR-1.7B", "family": "qwen"},
    # {"model_id": "Qwen/Qwen3-ASR-0.6B", "family": "qwen"},
]

print(f"Device: {DEVICE}")
print(f"Models to evaluate: {len(MODELS)}")

Device: cuda
Models to evaluate: 1


In [17]:
from datasets import load_dataset, Audio

print(f"[INFO] Loading dataset: {DATASET_ID} / {SPLIT}")

dataset = load_dataset(
    DATASET_ID,
    split="test",
    streaming=True
)

#override audio feature to prevent decoding
dataset = dataset.cast_column("audio", Audio(decode=False))

samples = list(dataset.take(NUM_SAMPLES))

[INFO] Loading dataset: IqraEval/QuranMB.v2 / test


In [18]:
ref_dataset = load_dataset(REF_DATASET, split="test")
ref_lookup  = {row["ID"]: row["Reference_phn"] for row in ref_dataset}

print(f"Loaded {len(samples)} audio samples, {len(ref_lookup)} reference entries")

Loaded 5 audio samples, 1642 reference entries


In [19]:
def normalise_arabic(text: str) -> str:
    """Strip diacritics and normalise Arabic text before scoring."""
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = re.sub(r"[إأآٱ]", "ا", text)
    text = text.replace("ـ", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def compute_per(references, hypotheses):
    return jiwer.wer(references, hypotheses)


def compute_bleu(references, hypotheses):
    return sacrebleu.corpus_bleu(hypotheses, [references], tokenize="13a")


def audio_to_wav_path(audio_array: np.ndarray, sr: int = TARGET_SR) -> str:
    """Write numpy audio to a temp WAV under /content (needed by some backends)."""
    fd, path = tempfile.mkstemp(suffix=".wav", dir=COLAB_WORKING)
    os.close(fd)
    sf.write(path, audio_array, sr)
    return path

In [30]:
# MODEL LOADERS & TRANSCRIBERS (per family)

def load_model(cfg: dict):
    family   = cfg["family"]
    model_id = cfg["model_id"]
    print(f"[LOAD] {model_id} ({family}) on {DEVICE}")

    if family == "omnilingual":
        from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
        pipeline = ASRInferencePipeline(model_card=cfg["model_card"], device=DEVICE)
        return {"family": family, "pipeline": pipeline}

    if family == "cohere":
        from transformers import AutoProcessor, CohereAsrForConditionalGeneration
        processor = AutoProcessor.from_pretrained(model_id)
        model = CohereAsrForConditionalGeneration.from_pretrained(
            model_id,
            device_map="auto" if DEVICE == "cuda" else None,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        )
        if DEVICE != "cuda":
            model = model.to(DEVICE)
        model.eval()
        return {"family": family, "processor": processor, "model": model}

    if family == "gemma":
        from transformers import AutoProcessor, AutoModelForCausalLM
        processor = AutoProcessor.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            dtype="auto",
            device_map="auto" if DEVICE == "cuda" else None,
            attn_implementation="sdpa",
        )
        if DEVICE != "cuda":
            model = model.to(DEVICE)
        model.eval()
        return {"family": family, "processor": processor, "model": model}

    if family == "qwen":
        from qwen_asr import Qwen3ASRModel
        model = Qwen3ASRModel.from_pretrained(
            model_id,
            dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
            device_map="cuda:0" if DEVICE == "cuda" else "cpu",
            max_inference_batch_size=1,
            max_new_tokens=256,
        )
        return {"family": family, "model": model}

    raise ValueError(f"Unknown family: {family}")


#def transcribe(handle: dict, audio_array: np.ndarray) -> str:
def transcribe(handle: dict, audio_array: np.ndarray, sampling_rate: int = 16000) -> str:
    # Now sampling_rate is defined and has a default value
    if sampling_rate != 16000:
        audio_array = librosa.resample(audio_array, orig_sr=sampling_rate, target_sr=16000)
    family = handle["family"]

    if family == "omnilingual":
        wav_path = audio_to_wav_path(audio_array)
        try:
            texts = handle["pipeline"].transcribe([wav_path], lang=["arb_Arab"], batch_size=1)
            return texts[0].strip()
        finally:
            os.remove(wav_path)

    if family == "cohere":
        processor, model = handle["processor"], handle["model"]
        inputs = processor(audio_array, sampling_rate=TARGET_SR, return_tensors="pt", language="ar")
        inputs = inputs.to(model.device, dtype=model.dtype)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=256)
        text = processor.decode(outputs, skip_special_tokens=True)
        return text[0].strip() if isinstance(text, list) else text.strip()

    if family == "gemma":
        # from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
        # processor = AutoProcessor.from_pretrained(model_id)

        # bnb_config = BitsAndBytesConfig(
        #     load_in_4bit=True,
        #     bnb_4bit_compute_dtype=torch.bfloat16,
        #     bnb_4bit_quant_type="nf4",
        #     bnb_4bit_use_double_quant=True,
        #     llm_int8_enable_fp32_cpu_offload=True,
        # ) if DEVICE == "cuda" else None

        # model = AutoModelForMultimodalLM.from_pretrained(
        #     model_id,
        #     quantization_config=bnb_config,
        #     dtype=torch.bfloat16 if DEVICE == "cuda" else "auto",
        #     device_map="auto" if DEVICE == "cuda" else None,
        #     attn_implementation="sdpa",
        # )
        # if DEVICE != "cuda":
        #     model = model.to(DEVICE)
        # model.eval()
        # return {"family": family, "processor": processor, "model": model}
        from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
        processor = AutoProcessor.from_pretrained(model_id)

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        ) if DEVICE == "cuda" else None

        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            dtype=torch.bfloat16 if DEVICE == "cuda" else "auto",
            device_map="auto" if DEVICE == "cuda" else None,
            max_memory={0: "13GiB", "cpu": "16GiB"} if DEVICE == "cuda" else None,
            attn_implementation="sdpa",
        )
        if DEVICE != "cuda":
            model = model.to(DEVICE)
        # NOTE: do not call model.to(...) when device_map="auto" is used on cuda —
        # the model is already dispatched via accelerate hooks; a manual .to()
        # is what produced the "Tensor on device meta" crash.
        model.eval()
        return {"family": family, "processor": processor, "model": model}
        import librosa
        processor, model = handle["processor"], handle["model"]

        # Resample to 16kHz if needed
        if sampling_rate != 16000:
            audio_array = librosa.resample(audio_array, orig_sr=sampling_rate, target_sr=16000)

        # Check audio length (max 30 seconds)
        duration = len(audio_array) / 16000
        if duration > 30:
            print(f"[WARNING] Audio {duration:.1f}s exceeds 30s limit, truncating")
            audio_array = audio_array[:30 * 16000]

        # Build proper ASR prompt (from Hugging Face model card)
        asr_prompt = """Transcribe the following speech segment in Arabic into Arabic text.

Follow these specific instructions for formatting the answer:
* Only output the transcription, with no newlines.
* When transcribing numbers, write the digits, i.e. write 1.7 and not one point seven, and write 3 instead of three."""

        # Process with text prompt + audio
        inputs = processor(
            text=asr_prompt,
            audio=audio_array,
            sampling_rate=16000,
            return_tensors="pt",
        )

        inputs = inputs.to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
            )

        # Decode response
        response = processor.decode(outputs[0], skip_special_tokens=True)

        # Extract just the transcription (remove prompt)
        text = response.split(asr_prompt)[-1].strip()
        return text
#         # # RECONSTRUCTED — source notebook cell was truncated here with no function body.
        # # Mirrors the cohere branch above (same transformers generate/decode pattern),
        # # adapted for AutoModelForMultimodalLM's expected audio input keys.
        # from transformers import AutoProcessor, AutoModelForCausalLM
        # processor = AutoProcessor.from_pretrained(model_id)
        # model = AutoModelForCausalLM.from_pretrained(
        #   model_id,
        #   dtype="auto",
        #   device_map="auto" if DEVICE == "cuda" else None,
        #   attn_implementation="sdpa",
        # )
        # if DEVICE != "cuda":
        #     model = model.to(DEVICE)
        # model.eval()
        # return {"family": family, "processor": processor, "model": model}
        # processor, model = handle["processor"], handle["model"]
        # inputs = processor(
        #     audio=audio_array,
        #     sampling_rate=TARGET_SR,
        #     text="<start_of_audio>",
        #     return_tensors="pt",
        # )
        # inputs = inputs.to(model.device, dtype=model.dtype)
        # with torch.no_grad():
        #     outputs = model.generate(**inputs, max_new_tokens=256)
        # text = processor.batch_decode(outputs, skip_special_tokens=True)
        # return text[0].strip() if isinstance(text, list) else text.strip()

    if family == "qwen":
        # RECONSTRUCTED — qwen-asr's high-level API typically exposes .transcribe()
        # directly on the model object taking a raw waveform array. Verify against
        # the installed qwen-asr version's actual signature before relying on this.
        model = handle["model"]
        result = model.transcribe(audio_array, sampling_rate=TARGET_SR)
        if isinstance(result, dict):
            return result.get("text", "").strip()
        return str(result).strip()

    raise ValueError(f"Unknown family: {family}")


def unload_model(handle: dict):
    # RECONSTRUCTED — not present in source notebook. Frees GPU memory between
    # models so later models in MODELS don't OOM on a single T4.
    for key in ("model", "pipeline"):
        obj = handle.get(key)
        if obj is not None and hasattr(obj, "to"):
            try:
                obj.to("cpu")
            except Exception:
                pass
    del handle
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [31]:
import io
import soundfile as sf
# EVALUATE ALL MODELS

all_results = []

for cfg in MODELS:
    model_id = cfg["model_id"]
    print("\n" + "=" * 70)
    print(f"EVALUATING: {model_id}")
    print("=" * 70)

    try:
        handle = load_model(cfg)
    except Exception as e:
        print(f"[ERROR] Failed to load {model_id}: {e}")
        all_results.append({"model_id": model_id, "status": "load_failed", "error": str(e)})
        continue

    references, hypotheses, sample_ids = [], [], []



    for idx, sample in enumerate(samples):
        audio_bytes = sample["audio"]["bytes"]
        audio_array, sampling_rate = sf.read(io.BytesIO(audio_bytes))
        sample_id   = sample["ID"]

        try:
            hyp_text     = transcribe(handle, audio_array, sampling_rate)
            hyp_text     = normalise_arabic(hyp_text)
            hyp_phonemes = arabic_to_phonemes(hyp_text)
        except Exception as e:
            print(f"  [{idx}] ID={sample_id} TRANSCRIBE ERROR: {e}")
            hyp_phonemes = ""

        ref_phonemes = ref_lookup.get(sample_id, "")
        references.append(ref_phonemes)
        hypotheses.append(hyp_phonemes)
        sample_ids.append(sample_id)

        print(f"\n  [{idx}] ID: {sample_id}")
        print(f"      REF: {ref_phonemes[:60]}...")
        print(f"      HYP: {hyp_phonemes[:60]}...")

    per  = compute_per(references, hypotheses)
    bleu = compute_bleu(references, hypotheses)

    result = {
        "model_id": model_id,
        "family": cfg["family"],
        "status": "ok",
        "corpus_per": per * 100,
        "corpus_bleu": bleu.score,
        "sample_ids": sample_ids,
        "references": references,
        "hypotheses": hypotheses,
    }
    all_results.append(result)

    print(f"\n  >> Corpus PER  : {per * 100:.2f}%")
    print(f"  >> Corpus BLEU : {bleu.score:.2f}")

    unload_model(handle)


EVALUATING: google/gemma-4-E2B-it
[LOAD] google/gemma-4-E2B-it (gemma) on cuda


[transformers] Current model requires 6178 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

  [0] ID=00000_00001 TRANSCRIBE ERROR: CUDA out of memory. Tried to allocate 3.89 GiB. GPU 0 has a total capacity of 14.56 GiB of which 2.25 GiB is free. Including non-PyTorch memory, this process has 12.31 GiB memory in use. Of the allocated memory 9.85 GiB is allocated by PyTorch, and 2.35 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

  [0] ID: 00000_00001
      REF: < i nn a m aa y a x $ aa ll a h a m i n E i b aa d i h i l E...
      HYP: ...
  [1] ID=00000_00002 TRANSCRIBE ERROR: CUDA out of memory. Tried to allocate 3.89 GiB. GPU 0 has a total capacity of 14.56 GiB of which 2.25 GiB is free. Including non-PyTorch memory, this process has 12.31 GiB memory in use. Of the allocated memory 9.85 GiB is allocated by PyTorch, and 2.35 GiB is reserve

In [24]:
# SUMMARY TABLE

rows = []
for r in all_results:
    if r["status"] == "ok":
        rows.append({
            "Model": r["model_id"],
            "Family": r["family"],
            "PER (%)": round(r["corpus_per"], 2),
            "BLEU": round(r["corpus_bleu"], 2),
        })
    else:
        rows.append({
            "Model": r["model_id"],
            "Family": "—",
            "PER (%)": "FAILED",
            "BLEU": r.get("error", "load error"),
        })

summary_df = pd.DataFrame(rows)
print("\n" + "=" * 70)
print("BENCHMARK SUMMARY")
print("=" * 70)
display(summary_df)


BENCHMARK SUMMARY


,Model,Family,PER (%),BLEU
0,google/gemma-4-E2B-it,gemma,100.0,0.0


In [25]:
# Per-sample PER breakdown for each successful model
for r in all_results:
    if r["status"] != "ok":
        continue
    print(f"\n--- {r['model_id']} ---")
    for i, (ref, hyp) in enumerate(zip(r["references"], r["hypotheses"])):
        p = compute_per([ref], [hyp])
        print(f"  [{i}] ID={r['sample_ids'][i]}  PER={p*100:.1f}%")

print("\nDone — designed for Colab T4/A100 GPU.")


--- google/gemma-4-E2B-it ---
  [0] ID=00000_00001  PER=100.0%
  [1] ID=00000_00002  PER=100.0%
  [2] ID=00000_00003  PER=100.0%
  [3] ID=00000_00004  PER=100.0%
  [4] ID=00000_00004439  PER=100.0%

Done — designed for Colab T4/A100 GPU.
